<a href="https://colab.research.google.com/github/arurion/Tools/blob/main/WAV2MIDIv2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title 【NEW!!!】音声・動画ファイル対応 180-TET MIDI Converter
#@markdown ---
#@markdown エラー要因（ピッチベンドの範囲指定）を修正しました。
#@markdown これで正しく音程補正がかかり、すべての倍音が出力されます。

#@markdown ### **パラメータ設定**

#@markdown **出力ファイル名**
OUTPUT_FILENAME = 'fft.mid' #@param {type:"string"}

#@markdown **楽曲のテンポ (BPM)**
BPM = 120 #@param {type:"integer"}

#@markdown **ベロシティ閾値 (dB)**
#@markdown -60推奨。ノイズが多い場合は-40くらいに上げてください。
VELOCITY_THRESHOLD_DB = -60 #@param {type:"slider", min:-80, max:-10, step:1}

#@markdown **FFTウィンドウサイズ**
N_FFT = 4096 #@param [2048, 4096, 8192] {type:"raw"}

#@markdown **ホップ長**
HOP_LENGTH = 256 #@param [128, 256, 512] {type:"raw"}
#@markdown ---

# 必要なライブラリとツールをインストール
!pip install librosa mido numpy scipy -q
!apt-get -qq install -y ffmpeg

import librosa
import numpy as np
from scipy.signal import find_peaks
import mido
from google.colab import files
import subprocess
import os
from pathlib import Path

def convert_audio_to_midi_fixed_v2(
    audio_file_path,
    output_filename='output.mid',
    bpm=120,
    velocity_threshold_db=-50,
    n_fft=4096,
    hop_length=256
):
    print(f"音声ファイル '{audio_file_path}' を読み込んでいます...")
    try:
        y, sr = librosa.load(audio_file_path, sr=None, mono=True)
    except Exception as e:
        print(f"エラー: 音声ファイルの読み込みに失敗しました。 {e}")
        return None

    print("STFT（短時間フーリエ変換）を実行中...")
    S = librosa.stft(y, n_fft=n_fft, hop_length=hop_length)
    magnitudes_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)
    freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)

    print("MIDIファイルを初期化しています...")
    # ドラムチャンネル(Ch 9 / index 9)を除く15チャンネルを使用
    channel_list = [i for i in range(16) if i != 9]

    mid = mido.MidiFile(ticks_per_beat=480, type=0)
    track = mido.MidiTrack()
    mid.tracks.append(track)

    tempo = mido.bpm2tempo(bpm)
    track.append(mido.MetaMessage('set_tempo', tempo=tempo, time=0))

    # --- チャンネル初期化とピッチベンド設定（修正版） ---

    pitch_bend_range_semitones = 2  # ピッチベンド幅（+/- 2半音）

    for idx, ch in enumerate(channel_list):
        # 1. 音色とRPN設定
        track.append(mido.Message('control_change', channel=ch, control=121, value=0, time=0)) # Reset
        track.append(mido.Message('program_change', channel=ch, program=80, time=0)) # Square Lead

        # Pitch Bend Range = 2 semitones
        track.append(mido.Message('control_change', channel=ch, control=101, value=0, time=0))
        track.append(mido.Message('control_change', channel=ch, control=100, value=0, time=0))
        track.append(mido.Message('control_change', channel=ch, control=6, value=pitch_bend_range_semitones, time=0))
        track.append(mido.Message('control_change', channel=ch, control=38, value=0, time=0))
        track.append(mido.Message('control_change', channel=ch, control=101, value=127, time=0))
        track.append(mido.Message('control_change', channel=ch, control=100, value=127, time=0))

        # 2. スタティック・ピッチベンドの設定 (ここを修正)
        # midoのpitchは -8192 〜 8191 (0が中心)

        # idxに応じて 0 〜 +100セント ずらす
        shift_cents = idx * (100.0 / 15.0)

        # 最大範囲(200セント)に対する割合
        ratio = shift_cents / (pitch_bend_range_semitones * 100.0)

        # mido用の値 (-8192 ~ 8191) に変換。
        # ここでは常にプラス方向（0〜8191）を使います。
        pitch_val = int(ratio * 8191)

        # 安全装置（範囲外エラー防止）
        pitch_val = max(-8192, min(8191, pitch_val))

        track.append(mido.Message('pitchwheel', channel=ch, pitch=pitch_val, time=0))

    print("全倍音のノートイベントを生成中...")
    events = []
    active_notes = {} # key: (channel, note), value: start_time_sec
    num_frames = S.shape[1]
    frame_time = hop_length / sr

    for frame_index in range(num_frames):
        if (frame_index + 1) % 500 == 0:
            print(f"  処理中: {frame_index + 1} / {num_frames} フレーム")

        current_time = frame_index * frame_time
        db_spec = magnitudes_db[:, frame_index]

        # ピーク検出（全倍音取得）
        peaks, _ = find_peaks(db_spec, height=velocity_threshold_db)

        detected_keys = set()

        for p in peaks:
            freq = freqs[p]
            if freq < 20: continue

            frac_midi = librosa.hz_to_midi(freq)

            # 180-TET マッピング
            micro_step = round(frac_midi * 15)
            base_note = int(micro_step // 15)
            micro_idx = int(micro_step % 15)

            if not (0 <= base_note <= 127): continue

            channel = channel_list[micro_idx]
            key = (channel, base_note)
            detected_keys.add(key)

            if key not in active_notes:
                amp_db = db_spec[p]
                velocity = int(np.interp(amp_db, [velocity_threshold_db, 0], [1, 127]))
                velocity = max(1, min(127, velocity))

                active_notes[key] = current_time
                events.append((current_time, mido.Message('note_on', channel=channel, note=base_note, velocity=velocity)))

        # ノートオフ処理
        active_keys_list = list(active_notes.keys())
        for key in active_keys_list:
            if key not in detected_keys:
                channel, base_note = key
                events.append((current_time, mido.Message('note_off', channel=channel, note=base_note, velocity=0)))
                del active_notes[key]

    end_time = num_frames * frame_time
    for key in active_notes:
        channel, base_note = key
        events.append((end_time, mido.Message('note_off', channel=channel, note=base_note, velocity=0)))

    print("イベントをソートして書き出し中...")
    events.sort(key=lambda x: x[0])

    last_time_ticks = 0
    for t_sec, msg in events:
        current_ticks = int(mido.second2tick(t_sec, mid.ticks_per_beat, tempo))
        delta = current_ticks - last_time_ticks
        if delta < 0: delta = 0
        msg.time = delta
        track.append(msg)
        last_time_ticks = current_ticks

    mid.save(output_filename)
    print(f"完了: '{output_filename}'")
    return output_filename

# --- Colab実行用コード ---
def run_in_colab():
    VIDEO_EXTENSIONS = ['.mp4', '.mov', '.mkv', '.webm', '.avi', '.flv']
    AUDIO_EXTENSIONS = ['.mp3', '.flac', '.wav', '.m4a', '.ogg', '.wma']

    print("変換したい音声ファイルまたは動画ファイルをアップロードしてください。")
    uploaded = files.upload()

    if not uploaded:
        print("ファイルがアップロードされませんでした。")
        return

    input_filename = next(iter(uploaded))
    uploaded_data = uploaded[input_filename]

    temp_input_path = "temp_input_file" + Path(input_filename).suffix
    temp_wav_path = "temp_audio.wav"
    audio_to_process = None

    try:
        with open(temp_input_path, "wb") as f:
            f.write(uploaded_data)

        file_ext = Path(temp_input_path).suffix.lower()

        if file_ext in VIDEO_EXTENSIONS:
            print(f"動画ファイル '{input_filename}' を検出しました。音声を抽出します...")
            command = ['ffmpeg', '-i', temp_input_path, '-vn', '-acodec', 'pcm_s16le', '-ar', '44100', '-ac', '1', temp_wav_path, '-y']
            subprocess.run(command, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
            audio_to_process = temp_wav_path

        elif file_ext in AUDIO_EXTENSIONS:
            print(f"音声ファイル '{input_filename}' を検出しました。")
            audio_to_process = temp_input_path

        else:
            print(f"エラー: 非対応のファイル形式です。")
            return

        output_file = convert_audio_to_midi_fixed_v2(
            audio_file_path=audio_to_process,
            output_filename=OUTPUT_FILENAME,
            bpm=BPM,
            velocity_threshold_db=VELOCITY_THRESHOLD_DB,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH
        )

        if output_file:
            print("\n変換が完了しました。ファイルをダウンロードします。")
            files.download(output_file)

    except Exception as e:
        print(f"エラーが発生しました: {e}")
    finally:
        if os.path.exists(temp_input_path): os.remove(temp_input_path)
        if os.path.exists(temp_wav_path): os.remove(temp_wav_path)

# 実行
run_in_colab()

In [ ]:
#@title 【OLD2】音声・動画ファイル対応 180-TET MIDI Converter
#@markdown ---
#@markdown MP3、MP4、MOVなど、様々な音声・動画ファイルから直接MIDIに変換します。

#@markdown ### **パラメータ設定**

#@markdown **出力ファイル名**
OUTPUT_FILENAME = 'final.mid' #@param {type:"string"}

#@markdown **楽曲のテンポ (BPM)**
BPM = 120 #@param {type:"integer"}

#@markdown **ベロシティ閾値 (dB)**
#@markdown このdB値より小さい音は無視されます。-60に近いほど小さな音も拾います。
VELOCITY_THRESHOLD_DB = -50 #@param {type:"slider", min:-70, max:-10, step:1}

#@markdown **FFTウィンドウサイズ**
#@markdown 周波数解像度。大きいほどピッチが正確になります。
N_FFT = 2048 #@param [2048, 4096, 8192] {type:"raw"}

#@markdown **ホップ長**
#@markdown 時間解像度。小さいほど速いパッセージに強くなります。
HOP_LENGTH = 256 #@param [128, 256, 512] {type:"raw"}
#@markdown ---

# 必要なライブラリとツールをインストール
!pip install librosa mido numpy scipy -q
!apt-get -qq install -y ffmpeg

import librosa
import numpy as np
from scipy.signal import find_peaks
import mido
from google.colab import files
import subprocess
import os
from pathlib import Path

def convert_audio_to_midi(
    audio_file_path,
    output_filename='output.mid',
    bpm=120,
    velocity_threshold_db=-50,
    n_fft=2048,
    hop_length=256
):
    """
    オーディオファイルを読み込み、高品位なMIDIに変換するコア関数
    """
    print(f"音声ファイル '{audio_file_path}' を読み込んでいます...")
    try:
        y, sr = librosa.load(audio_file_path, sr=None, mono=True)
    except Exception as e:
        print(f"エラー: 音声ファイルの読み込みに失敗しました。 {e}")
        return

    print("STFT（短時間フーリエ変換）を実行中...")
    S = librosa.stft(y, n_fft=n_fft, hop_length=hop_length)
    freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
    magnitudes_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)

    print("MIDIファイルを初期化しています...")
    channel_list = [i for i in range(16) if i != 9]
    mid = mido.MidiFile(ticks_per_beat=480, type=0)
    track = mido.MidiTrack()
    mid.tracks.append(track)
    tempo = mido.bpm2tempo(bpm)
    track.append(mido.MetaMessage('set_tempo', tempo=tempo, time=0))

    print("MIDIチャンネルを設定しています...")
    for ch in channel_list:
        track.append(mido.Message('control_change', channel=ch, control=0, value=0, time=0))
        track.append(mido.Message('control_change', channel=ch, control=32, value=2, time=0))
        track.append(mido.Message('program_change', channel=ch, program=80, time=0))
        track.append(mido.Message('control_change', channel=ch, control=101, value=0, time=0))
        track.append(mido.Message('control_change', channel=ch, control=100, value=0, time=0))
        track.append(mido.Message('control_change', channel=ch, control=6, value=2, time=0))
        track.append(mido.Message('control_change', channel=ch, control=38, value=0, time=0))
        track.append(mido.Message('control_change', channel=ch, control=101, value=127, time=0))
        track.append(mido.Message('control_change', channel=ch, control=100, value=127, time=0))

    range_cents = 200.0
    step_cents = 100.0 / 15.0
    for idx, ch in enumerate(channel_list):
        shift_cents = idx * step_cents - 50.0 + (step_cents / 2.0)
        pitch_value = round((shift_cents / range_cents) * 8192)
        track.append(mido.Message('pitchwheel', channel=ch, pitch=pitch_value, time=0))

    print("ノートイベントを生成しています...")
    events = []
    active_notes = {}
    num_frames = S.shape[1]
    for frame_index in range(num_frames):
        if (frame_index + 1) % 500 == 0:
            print(f"  進捗: {frame_index + 1} / {num_frames} フレーム")

        time_sec = frame_index * hop_length / sr
        db_spec_frame = magnitudes_db[:, frame_index]

        peaks, _ = find_peaks(db_spec_frame, height=velocity_threshold_db)
        detected_this_frame = set()

        for p in peaks:
            freq = freqs[p]
            if freq <= 0: continue

            frac_midi = librosa.hz_to_midi(freq)
            if frac_midi is None or not (0 <= frac_midi < 128): continue

            micro_step = round(frac_midi * 15)
            base_note = micro_step // 15
            micro_idx = micro_step % 15

            if not (0 <= base_note <= 127): continue

            channel = channel_list[micro_idx]
            key = (channel, base_note)
            detected_this_frame.add(key)

            if key not in active_notes:
                db_val = db_spec_frame[p]
                velocity = int(np.interp(db_val, [velocity_threshold_db, 0], [1, 127]))
                velocity = max(1, min(127, velocity))
                active_notes[key] = time_sec
                msg_on = mido.Message('note_on', channel=channel, note=base_note, velocity=velocity)
                events.append((time_sec, msg_on))

        released_notes = set(active_notes.keys()) - detected_this_frame
        for key in released_notes:
            channel, base_note = key
            msg_off = mido.Message('note_off', channel=channel, note=base_note, velocity=0)
            events.append((time_sec, msg_off))
            del active_notes[key]

    end_time_sec = len(y) / sr
    for key in list(active_notes):
        channel, base_note = key
        msg_off = mido.Message('note_off', channel=channel, note=base_note, velocity=0)
        events.append((end_time_sec, msg_off))

    print("MIDIイベントを時間順にソートしています...")
    events.sort(key=lambda e: e[0])

    current_time_ticks = 0
    for time_sec, msg in events:
        abs_ticks = int(mido.second2tick(time_sec, mid.ticks_per_beat, tempo))
        delta_ticks = abs_ticks - current_time_ticks
        msg.time = delta_ticks
        track.append(msg)
        current_time_ticks = abs_ticks

    print("MIDIファイルを保存しています...")
    mid.save(output_filename)
    print(f"変換完了: '{output_filename}'")
    return output_filename

# --- Colab実行用コード ---
def run_in_colab():
    # 対応するファイル拡張子
    VIDEO_EXTENSIONS = ['.mp4', '.mov', '.mkv', '.webm', '.avi', '.flv']
    AUDIO_EXTENSIONS = ['.mp3', '.flac', '.wav', '.m4a', '.ogg', '.wma']

    print("変換したい音声ファイルまたは動画ファイルをアップロードしてください。")
    uploaded = files.upload()

    if not uploaded:
        print("ファイルがアップロードされませんでした。")
        return

    input_filename = next(iter(uploaded))
    uploaded_data = uploaded[input_filename]

    # 一時ファイル名を定義
    temp_input_path = "temp_input_file" + Path(input_filename).suffix
    temp_wav_path = "temp_audio.wav"
    audio_to_process = None

    try:
        # アップロードされたデータを一時ファイルとして保存
        with open(temp_input_path, "wb") as f:
            f.write(uploaded_data)

        file_ext = Path(temp_input_path).suffix.lower()

        if file_ext in VIDEO_EXTENSIONS:
            print(f"動画ファイル '{input_filename}' を検出しました。音声を抽出します...")
            # ffmpegコマンド: -i 入力, -vn ビデオなし, -acodec pcm_s16le 高品質WAV, -ar 44100 サンプルレート, -ac 1 モノラル
            command = ['ffmpeg', '-i', temp_input_path, '-vn', '-acodec', 'pcm_s16le', '-ar', '44100', '-ac', '1', temp_wav_path, '-y']
            subprocess.run(command, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
            audio_to_process = temp_wav_path
            print("音声の抽出が完了しました。")

        elif file_ext in AUDIO_EXTENSIONS:
            print(f"音声ファイル '{input_filename}' を検出しました。")
            audio_to_process = temp_input_path

        else:
            print(f"エラー: 非対応のファイル形式です。 ({file_ext})")
            return

        # MIDI変換を実行
        output_file = convert_audio_to_midi(
            audio_file_path=audio_to_process,
            output_filename=OUTPUT_FILENAME,
            bpm=BPM,
            velocity_threshold_db=VELOCITY_THRESHOLD_DB,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH
        )

        if output_file:
            print("\n変換が完了しました。ファイルをダウンロードします。")
            files.download(output_file)

    except subprocess.CalledProcessError as e:
        print("エラー: FFmpegでの音声抽出に失敗しました。")
        print(f"FFmpeg stderr: {e.stderr.decode('utf-8')}")
    except Exception as e:
        print(f"予期せぬエラーが発生しました: {e}")
    finally:
        # 一時ファイルをクリーンアップ
        print("一時ファイルを削除しています...")
        if os.path.exists(temp_input_path):
            os.remove(temp_input_path)
        if os.path.exists(temp_wav_path):
            os.remove(temp_wav_path)

# 実行
run_in_colab()

下は旧版です

In [ ]:
#@title 【OLD】WAV to 180-TET MIDI Converter (高品位ロジック版)
#@markdown ---
#@markdown ご提示いただいた優れたロジックをベースに、Colabでの使いやすさを向上させました。

#@markdown ### **パラメータ設定**

#@markdown **出力ファイル名**
OUTPUT_FILENAME = '未定.mid' #@param {type:"string"}

#@markdown **楽曲のテンポ (BPM)**
BPM = 120 #@param {type:"integer"}

#@markdown **ベロシティ閾値 (dB)**
#@markdown このdB値より小さい音は無視されます。-60に近いほど小さな音も拾います。
VELOCITY_THRESHOLD_DB = -50 #@param {type:"slider", min:-70, max:-10, step:1}

#@markdown **FFTウィンドウサイズ**
#@markdown 周波数解像度。大きいほどピッチが正確になります。
N_FFT = 2048 #@param [2048, 4096, 8192] {type:"raw"}

#@markdown **ホップ長**
#@markdown 時間解像度。小さいほど速いパッセージに強くなります。
HOP_LENGTH = 256 #@param [128, 256, 512] {type:"raw"}
#@markdown ---

# 必要なライブラリをインストール
!pip install librosa mido numpy scipy

import librosa
import numpy as np
from scipy.signal import find_peaks
import mido
from google.colab import files
import io

def convert_wav_to_midi_improved(
    wav_data,
    output_filename='output.mid',
    bpm=120,
    velocity_threshold_db=-50,
    n_fft=2048,
    hop_length=256
):
    """
    提示された優れたロジックをベースにした変換関数
    """
    print("WAVファイルを読み込んでいます...")
    try:
        y, sr = librosa.load(io.BytesIO(wav_data), sr=None, mono=True)
    except Exception as e:
        print(f"エラー: WAVファイルの読み込みに失敗しました。 {e}")
        return

    print("STFTを実行中...")
    S = librosa.stft(y, n_fft=n_fft, hop_length=hop_length)
    freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
    magnitudes_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)

    print("MIDIファイルを初期化しています...")
    # MIDIチャンネルリスト（チャンネル10をスキップ）
    channel_list = [i for i in range(16) if i != 9]

    # MIDIファイル作成 (Type 0)
    mid = mido.MidiFile(ticks_per_beat=480, type=0)
    track = mido.MidiTrack()
    mid.tracks.append(track)

    # テンポ設定
    tempo = mido.bpm2tempo(bpm)
    track.append(mido.MetaMessage('set_tempo', tempo=tempo, time=0))

    # 各チャンネルの初期設定
    print("MIDIチャンネルを設定しています...")
    for ch in channel_list:
        # Bank Select Bank3(内部値2): MSB=0, LSB=2
        track.append(mido.Message('control_change', channel=ch, control=0, value=0, time=0))
        track.append(mido.Message('control_change', channel=ch, control=32, value=2, time=0))
        # Program Change 81 (Saw Wave)
        track.append(mido.Message('program_change', channel=ch, program=80, time=0))
        # Pitch Bend Sensitivity (RPN 0: ±2 semitones)
        track.append(mido.Message('control_change', channel=ch, control=101, value=0, time=0))
        track.append(mido.Message('control_change', channel=ch, control=100, value=0, time=0))
        track.append(mido.Message('control_change', channel=ch, control=6, value=2, time=0))
        track.append(mido.Message('control_change', channel=ch, control=38, value=0, time=0))
        track.append(mido.Message('control_change', channel=ch, control=101, value=127, time=0))
        track.append(mido.Message('control_change', channel=ch, control=100, value=127, time=0))

    # 各チャンネルに固定ピッチベンドを設定（±2半音レンジ内で+0~+1半音弱をカバー）
    range_cents = 200.0  # ピッチベンドレンジ (±2半音 = 200セント)
    step_cents = 100.0 / 15.0 # 1半音を15分割したステップ（セント単位）
    for idx, ch in enumerate(channel_list):
        # チャンネルindexに応じてピッチをずらす
        shift_cents = idx * step_cents - 50.0 + (step_cents / 2.0)
        # pitch_value: -8192..+8191
        pitch_value = round((shift_cents / range_cents) * 8192)
        track.append(mido.Message('pitchwheel', channel=ch, pitch=pitch_value, time=0))

    # --- イベント生成 ---
    print("ノートイベントを生成しています...")
    events = []
    active_notes = {}  # (ch, note): start_time_sec

    num_frames = S.shape[1]
    for frame_index in range(num_frames):
        if (frame_index + 1) % 500 == 0:
            print(f"  進捗: {frame_index + 1} / {num_frames} フレーム")

        time_sec = frame_index * hop_length / sr
        db_spec_frame = magnitudes_db[:, frame_index]

        # ピーク検出
        peaks, props = find_peaks(db_spec_frame, height=velocity_threshold_db)
        detected_this_frame = set()

        for p in peaks:
            freq = freqs[p]
            if freq <= 0: continue

            # 周波数を小数点付きMIDIノート番号に変換
            frac_midi = librosa.hz_to_midi(freq)
            if frac_midi is None or not (0 <= frac_midi < 128): continue

            # 180-TETステップに丸め込み
            micro_step = round(frac_midi * 15)
            base_note = micro_step // 15
            micro_idx = micro_step % 15

            if not (0 <= base_note <= 127): continue

            channel = channel_list[micro_idx]
            key = (channel, base_note)
            detected_this_frame.add(key)

            # 新しいノートを検出した場合: Note On
            if key not in active_notes:
                db_val = db_spec_frame[p]
                velocity = int(np.interp(db_val, [velocity_threshold_db, 0], [1, 127]))
                velocity = max(1, min(127, velocity))

                active_notes[key] = time_sec
                msg_on = mido.Message('note_on', channel=channel, note=base_note, velocity=velocity)
                events.append((time_sec, msg_on))

        # 鳴り止んだノートを検出した場合: Note Off
        released_notes = set(active_notes.keys()) - detected_this_frame
        for key in released_notes:
            channel, base_note = key
            msg_off = mido.Message('note_off', channel=channel, note=base_note, velocity=0)
            # Note Offの時間は現在のフレーム時間
            events.append((time_sec, msg_off))
            del active_notes[key]

    # 曲の最後に残っているノートをすべてオフにする
    end_time_sec = len(y) / sr
    for key in list(active_notes):
        channel, base_note = key
        msg_off = mido.Message('note_off', channel=channel, note=base_note, velocity=0)
        events.append((end_time_sec, msg_off))

    # --- イベントを時間順にソートし、デルタタイムを計算してトラックに追加 ---
    print("MIDIイベントを時間順にソートしています...")
    events.sort(key=lambda e: e[0])

    current_time_ticks = 0
    for time_sec, msg in events:
        abs_ticks = int(mido.second2tick(time_sec, mid.ticks_per_beat, tempo))
        delta_ticks = abs_ticks - current_time_ticks
        msg.time = delta_ticks
        track.append(msg)
        current_time_ticks = abs_ticks

    # MIDI保存
    print("MIDIファイルを保存しています...")
    mid.save(output_filename)
    print(f"変換完了: '{output_filename}'")
    return output_filename

# --- Colab実行用コード ---
def run_in_colab():
    print("変換したいWAVファイルをアップロードしてください。")
    uploaded = files.upload()

    if not uploaded:
        print("ファイルがアップロードされませんでした。")
        return

    input_wav_name = next(iter(uploaded))
    wav_data = uploaded[input_wav_name]

    print(f"\nファイル '{input_wav_name}' の変換を開始します。")

    output_file = convert_wav_to_midi_improved(
        wav_data=wav_data,
        output_filename=OUTPUT_FILENAME,
        bpm=BPM,
        velocity_threshold_db=VELOCITY_THRESHOLD_DB,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH
    )

    if output_file:
        print("\n変換が完了しました。ファイルをダウンロードします。")
        files.download(output_file)

# 実行
run_in_colab()